# Module 1, Notebook 5: File I/O and Error Handling

## For Senior Swift Engineers

This notebook covers Python's approach to file operations, path handling, data serialization (JSON/CSV), exception handling, and logging. As an iOS engineer, you have used `FileManager`, `Codable`, `do/catch`, and `os.log` -- Python has equivalents for all of these, with a more dynamic and convention-driven flavor.

---

## Swift vs Python Quick Reference

| Concept | Swift | Python |
|---|---|---|
| File reading | `String(contentsOfFile:)` / `FileManager` | `open()` + `.read()` / `.readlines()` |
| File writing | `string.write(toFile:)` / `FileManager` | `open(mode='w')` + `.write()` |
| Path handling | `URL` / `FileManager` | `pathlib.Path` |
| Path joining | `url.appendingPathComponent()` | `Path() / "subdir" / "file.txt"` |
| JSON decoding | `JSONDecoder().decode(T.self, from:)` | `json.loads(string)` / `json.load(file)` |
| JSON encoding | `JSONEncoder().encode(object)` | `json.dumps(obj)` / `json.dump(obj, file)` |
| CSV reading | No built-in (use libraries) | `csv.reader` / `csv.DictReader` |
| Error handling | `do { try } catch { }` | `try: except:` |
| Throwing functions | `func foo() throws -> T` | No declaration needed (any function can raise) |
| Custom errors | `enum MyError: Error { }` | `class MyError(Exception): pass` |
| Cleanup | `defer { }` | `finally:` or `with` statement |
| Logging | `os.log` / `Logger` | `logging` module |

---

## 1. File Operations

Python's built-in `open()` function is your primary tool for file I/O. It returns a file object that you use within a `with` block (context manager) to guarantee the file gets closed.

### Swift comparison

```swift
// Swift: reading a file
let content = try String(contentsOfFile: "/path/to/file.txt", encoding: .utf8)

// Swift: writing a file
try "Hello".write(toFile: "/path/to/file.txt", atomically: true, encoding: .utf8)
```

In [ ]:
# Writing files
import tempfile
import os

# Create a temporary directory for our examples
demo_dir = tempfile.mkdtemp(prefix="python_io_demo_")
print(f"Working in: {demo_dir}")

# open() modes:
# 'r'  - read (default)
# 'w'  - write (creates file, truncates if exists)
# 'a'  - append (creates file if doesn't exist)
# 'x'  - exclusive creation (fails if file exists)
# 'b'  - binary mode (e.g., 'rb', 'wb')
# 't'  - text mode (default, e.g., 'rt' same as 'r')
# '+'  - read and write (e.g., 'r+')

filepath = os.path.join(demo_dir, "sample.txt")

# Write a file
with open(filepath, 'w') as f:
    f.write("Line 1: Hello, Python!\n")
    f.write("Line 2: File I/O is straightforward.\n")
    f.write("Line 3: Context managers handle cleanup.\n")

print(f"File written to: {filepath}")

In [ ]:
# Reading files — multiple approaches

# 1. Read entire file as a single string
with open(filepath, 'r') as f:
    content = f.read()
print("--- read() ---")
print(content)

# 2. Read all lines as a list
with open(filepath, 'r') as f:
    lines = f.readlines()  # includes \n at end of each line
print("--- readlines() ---")
print(lines)

# 3. Iterate line by line (memory efficient — best for large files)
print("--- line by line ---")
with open(filepath, 'r') as f:
    for line_number, line in enumerate(f, start=1):
        print(f"{line_number}: {line.rstrip()}")  # rstrip() removes trailing \n

In [ ]:
# Appending to a file

with open(filepath, 'a') as f:
    f.write("Line 4: Appended later.\n")

with open(filepath, 'r') as f:
    print(f.read())

# Writing multiple lines at once with writelines()
new_file = os.path.join(demo_dir, "writelines_demo.txt")
lines_to_write = ["Alpha\n", "Beta\n", "Gamma\n"]

with open(new_file, 'w') as f:
    f.writelines(lines_to_write)  # NOTE: writelines does NOT add \n automatically

with open(new_file, 'r') as f:
    print(f.read())

In [ ]:
# Binary file operations (images, PDFs, etc.)
# In Swift: Data(contentsOf: url)

binary_path = os.path.join(demo_dir, "binary_demo.bin")

# Write binary data
with open(binary_path, 'wb') as f:
    f.write(b'\x00\x01\x02\x03')       # bytes literal
    f.write(bytes([255, 254, 253]))      # from list of ints
    f.write("Hello".encode('utf-8'))     # encode string to bytes

# Read binary data
with open(binary_path, 'rb') as f:
    data = f.read()
    print(f"Raw bytes: {data}")
    print(f"Hex: {data.hex()}")
    print(f"Length: {len(data)} bytes")

---

## 2. Path Handling with `pathlib`

`pathlib.Path` is Python's modern, object-oriented path library. It is to Python what `URL`/`FileManager` is to Swift, but more ergonomic.

### Swift comparison

```swift
// Swift: path operations with URL
let homeDir = FileManager.default.homeDirectoryForCurrentUser
let configPath = homeDir
    .appendingPathComponent(".config")
    .appendingPathComponent("app.json")
let exists = FileManager.default.fileExists(atPath: configPath.path)
```

In [ ]:
from pathlib import Path

# Creating paths — the / operator joins path components (like appendingPathComponent)
home = Path.home()
config_path = home / ".config" / "app.json"
print(f"Config path: {config_path}")

# Path from string
p = Path("/usr/local/bin/python3")

# Path components (like URL properties in Swift)
print(f"Name: {p.name}")           # python3
print(f"Stem: {p.stem}")           # python3
print(f"Suffix: {p.suffix}")       # (empty — no extension)
print(f"Parent: {p.parent}")       # /usr/local/bin
print(f"Parts: {p.parts}")         # ('/', 'usr', 'local', 'bin', 'python3')

# File with extension
doc = Path("/Users/daniel/report.pdf")
print(f"\nName: {doc.name}")       # report.pdf
print(f"Stem: {doc.stem}")         # report
print(f"Suffix: {doc.suffix}")     # .pdf

# Change extension (like Swift URL.deletingPathExtension().appendingPathExtension())
txt_doc = doc.with_suffix('.txt')
print(f"Changed extension: {txt_doc}")  # /Users/daniel/report.txt

# Change filename
new_doc = doc.with_name('summary.pdf')
print(f"Changed name: {new_doc}")  # /Users/daniel/summary.pdf

In [ ]:
from pathlib import Path

# Checking existence and type (like FileManager.fileExists)
p = Path(demo_dir)
print(f"Exists: {p.exists()}")        # True
print(f"Is file: {p.is_file()}")      # False (it's a directory)
print(f"Is dir: {p.is_dir()}")        # True

sample = Path(filepath)
print(f"\n{sample.name} exists: {sample.exists()}")   # True
print(f"{sample.name} is file: {sample.is_file()}")   # True

# Resolve to absolute path (like URL.standardized)
relative = Path(".") / "some" / "path"
print(f"\nRelative: {relative}")
print(f"Resolved: {relative.resolve()}")

# Current working directory
print(f"\nCWD: {Path.cwd()}")

In [ ]:
from pathlib import Path

# Directory operations
demo = Path(demo_dir)

# Create nested directories (like FileManager.createDirectory(withIntermediateDirectories:))
nested = demo / "subdir" / "deep"
nested.mkdir(parents=True, exist_ok=True)
print(f"Created: {nested}")

# Create a few files for demonstration
(demo / "data.json").write_text('{"key": "value"}')
(demo / "data.csv").write_text('name,age\nAlice,30')
(demo / "subdir" / "notes.txt").write_text('Some notes')
(demo / "subdir" / "deep" / "config.json").write_text('{}')

# List directory contents (like FileManager.contentsOfDirectory)
print("\nDirect children:")
for item in sorted(demo.iterdir()):
    kind = "DIR" if item.is_dir() else "FILE"
    print(f"  [{kind}] {item.name}")

# Glob pattern matching (like FileManager with NSPredicate)
print("\n*.json files (non-recursive):")
for f in demo.glob("*.json"):
    print(f"  {f}")

# Recursive glob with **
print("\n*.json files (recursive):")
for f in demo.rglob("*.json"):
    print(f"  {f}")

In [ ]:
# pathlib can also read and write files directly
# Convenient for small files

from pathlib import Path

p = Path(demo_dir) / "quick_demo.txt"

# Write (creates the file)
p.write_text("Quick and easy file writing!\nLine two.")

# Read
content = p.read_text()
print(content)

# Binary versions
p_bin = Path(demo_dir) / "quick_binary.bin"
p_bin.write_bytes(b'\x00\x01\x02')
data = p_bin.read_bytes()
print(f"Binary: {data}")

# File stats
stat = p.stat()
print(f"\nSize: {stat.st_size} bytes")
print(f"Modified: {stat.st_mtime}")

---

## 3. JSON Handling

Python's `json` module is the equivalent of Swift's `JSONEncoder`/`JSONDecoder` + `Codable`. The key difference: Python works with raw dicts/lists, while Swift decodes into strongly-typed structs.

### Swift comparison

```swift
// Swift: strongly typed with Codable
struct User: Codable {
    let name: String
    let age: Int
}

// Decode
let user = try JSONDecoder().decode(User.self, from: jsonData)
print(user.name)  // type-safe access

// Encode
let data = try JSONEncoder().encode(user)
```

```python
# Python: dynamic dicts
user = json.loads(json_string)  # returns a dict
print(user["name"])  # string key access, no type safety
```

In [ ]:
import json

# --- Parsing JSON (like JSONDecoder().decode()) ---

json_string = '''
{
    "name": "Daniel",
    "age": 30,
    "skills": ["Swift", "Python", "iOS"],
    "address": {
        "city": "San Francisco",
        "state": "CA"
    },
    "active": true
}
'''

# json.loads() — parse a JSON string ("loads" = load string)
data = json.loads(json_string)
print(type(data))          # <class 'dict'>
print(data["name"])        # Daniel
print(data["skills"][0])   # Swift
print(data["address"]["city"])  # San Francisco

# JSON types map to Python types:
# JSON object  ->  dict
# JSON array   ->  list
# JSON string  ->  str
# JSON number  ->  int or float
# JSON true    ->  True
# JSON false   ->  False
# JSON null    ->  None

In [ ]:
import json

# --- Generating JSON (like JSONEncoder().encode()) ---

user = {
    "name": "Daniel",
    "age": 30,
    "skills": ["Swift", "Python"],
    "active": True,
    "score": None,  # None becomes null in JSON
}

# json.dumps() — serialize to a JSON string ("dumps" = dump string)
json_output = json.dumps(user)
print("Compact:")
print(json_output)

# Pretty printing (like JSONEncoder.OutputFormatting.prettyPrinted)
json_pretty = json.dumps(user, indent=2, sort_keys=True)
print("\nPretty:")
print(json_pretty)

In [ ]:
import json
from pathlib import Path

# --- File-based JSON operations ---

json_path = Path(demo_dir) / "users.json"

users = [
    {"name": "Alice", "age": 28, "role": "engineer"},
    {"name": "Bob", "age": 35, "role": "designer"},
    {"name": "Charlie", "age": 42, "role": "manager"},
]

# json.dump() — write directly to a file (not dumps)
with open(json_path, 'w') as f:
    json.dump(users, f, indent=2)

print(f"Written to {json_path}")

# json.load() — read directly from a file (not loads)
with open(json_path, 'r') as f:
    loaded_users = json.load(f)

for user in loaded_users:
    print(f"{user['name']} ({user['role']}), age {user['age']}")

In [ ]:
# Making custom objects JSON-serializable
# In Swift, you'd conform to Codable. In Python, you need a custom encoder.

import json
from datetime import datetime
from dataclasses import dataclass, asdict

@dataclass
class Event:
    title: str
    date: datetime
    attendees: list[str]

event = Event(
    title="Python Workshop",
    date=datetime(2025, 6, 15, 14, 0),
    attendees=["Alice", "Bob"],
)

# This would fail: json.dumps(event)  # TypeError: Object is not JSON serializable

# Solution 1: Custom encoder
class EventEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        if hasattr(obj, '__dataclass_fields__'):
            return asdict(obj)
        return super().default(obj)

json_str = json.dumps(event, cls=EventEncoder, indent=2)
print(json_str)

# Solution 2: Simpler — use a default function
def json_serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

json_str2 = json.dumps(asdict(event), default=json_serializer, indent=2)
print(json_str2)

---

## 4. CSV Handling

Swift has no built-in CSV support (you typically use libraries like CodableCSV). Python has the `csv` module in the standard library.

In [ ]:
import csv
from pathlib import Path

csv_path = Path(demo_dir) / "employees.csv"

# --- Writing CSV ---

# csv.writer — writes lists as rows
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["name", "department", "salary"])  # header
    writer.writerow(["Alice", "Engineering", 120000])
    writer.writerow(["Bob", "Design", 95000])
    writer.writerow(["Charlie", "Engineering", 135000])
    writer.writerow(["Diana", "Marketing", 88000])

print(f"CSV written to {csv_path}")
print(csv_path.read_text())

In [ ]:
import csv

# --- Reading CSV ---

# csv.reader — reads rows as lists
print("--- csv.reader (list rows) ---")
with open(csv_path, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)  # first row is the header
    print(f"Header: {header}")
    for row in reader:
        print(f"  {row[0]} in {row[1]}, salary ${row[2]}")

print()

# csv.DictReader — reads rows as dicts (using header as keys)
# This is usually what you want — much more readable
print("--- csv.DictReader (dict rows) ---")
with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # row is an OrderedDict: {'name': 'Alice', 'department': 'Engineering', ...}
        print(f"  {row['name']} in {row['department']}, salary ${row['salary']}")

In [ ]:
import csv
from pathlib import Path

# csv.DictWriter — write dicts as rows
# Great for when your data is already in dict form

output_path = Path(demo_dir) / "output.csv"

employees = [
    {"name": "Eve", "department": "Engineering", "salary": 110000},
    {"name": "Frank", "department": "Sales", "salary": 92000},
]

with open(output_path, 'w', newline='') as f:
    fieldnames = ["name", "department", "salary"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(employees)  # write all rows at once

print(output_path.read_text())

---

## 5. Exception Handling

Python's exception handling is conceptually similar to Swift's `do/catch`, but with key differences:

1. **Any function can raise** — there is no `throws` keyword to declare it
2. **Exceptions are classes**, not enums
3. **Python convention: EAFP** (Easier to Ask Forgiveness than Permission) vs Swift's LBYL (Look Before You Leap)

### Swift comparison

```swift
// Swift: must declare throws, must use try
func parse(_ s: String) throws -> Int {
    guard let n = Int(s) else { throw ParseError.invalidInput(s) }
    return n
}

do {
    let n = try parse("abc")
} catch ParseError.invalidInput(let s) {
    print("Bad input: \(s)")
} catch {
    print("Unknown error: \(error)")
}
```

In [ ]:
# try / except / else / finally

def divide(a: float, b: float) -> float:
    """Divide a by b. No 'throws' needed — any function can raise."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

# Full exception handling syntax:
try:
    result = divide(10, 3)
except ValueError as e:
    # Like Swift's 'catch ValueErrror.xxx'
    print(f"Value error: {e}")
except TypeError as e:
    # Can have multiple except blocks (like multiple catch clauses)
    print(f"Type error: {e}")
except Exception as e:
    # Catch-all (like Swift's 'catch' without a pattern)
    print(f"Unexpected error: {e}")
else:
    # Runs ONLY if no exception occurred (no Swift equivalent)
    print(f"Result: {result:.4f}")
finally:
    # Always runs (like Swift's defer, but specifically for try blocks)
    print("Division attempted.")

In [ ]:
# Exception hierarchy — all exceptions inherit from BaseException
# You should typically catch Exception (not BaseException)

# BaseException
#  +-- SystemExit          (sys.exit())
#  +-- KeyboardInterrupt   (Ctrl+C)
#  +-- Exception           <-- catch this or its subclasses
#       +-- ValueError
#       +-- TypeError
#       +-- KeyError
#       +-- IndexError
#       +-- FileNotFoundError
#       +-- IOError
#       +-- RuntimeError
#       +-- StopIteration
#       +-- AttributeError
#       ... and many more

# Common exceptions you'll encounter:
examples = {
    "KeyError": lambda: {}["missing_key"],
    "IndexError": lambda: [1, 2, 3][10],
    "TypeError": lambda: "hello" + 42,
    "ValueError": lambda: int("not_a_number"),
    "AttributeError": lambda: None.some_method(),
    "ZeroDivisionError": lambda: 1 / 0,
    "FileNotFoundError": lambda: open("/nonexistent/file.txt"),
}

for name, func in examples.items():
    try:
        func()
    except Exception as e:
        print(f"{name}: {e}")

In [ ]:
# Custom exceptions — like Swift's custom Error types

# Swift:
# enum NetworkError: Error {
#     case timeout
#     case unauthorized
#     case serverError(statusCode: Int)
# }

# Python: use class hierarchy
class AppError(Exception):
    """Base exception for our application."""
    pass

class NetworkError(AppError):
    """Base class for network-related errors."""
    pass

class TimeoutError(NetworkError):
    """Request timed out."""
    def __init__(self, url: str, timeout_seconds: float):
        self.url = url
        self.timeout_seconds = timeout_seconds
        super().__init__(f"Request to {url} timed out after {timeout_seconds}s")

class UnauthorizedError(NetworkError):
    """Authentication failed."""
    def __init__(self, url: str):
        self.url = url
        super().__init__(f"Unauthorized access to {url}")

class ServerError(NetworkError):
    """Server returned an error."""
    def __init__(self, url: str, status_code: int):
        self.url = url
        self.status_code = status_code
        super().__init__(f"Server error {status_code} from {url}")

class ValidationError(AppError):
    """Data validation failed."""
    def __init__(self, field: str, message: str):
        self.field = field
        super().__init__(f"Validation error on '{field}': {message}")

# Usage
def fetch_data(url: str):
    """Simulate fetching data that might fail."""
    if "timeout" in url:
        raise TimeoutError(url, 30.0)
    if "auth" in url:
        raise UnauthorizedError(url)
    if "500" in url:
        raise ServerError(url, 500)
    return {"data": "success"}

# Catching at different levels of the hierarchy
for url in ["https://api.example.com/timeout", "https://api.example.com/auth", "https://api.example.com/data"]:
    try:
        result = fetch_data(url)
        print(f"Success: {result}")
    except TimeoutError as e:
        print(f"Timeout: {e.url} after {e.timeout_seconds}s")
    except NetworkError as e:
        # Catches UnauthorizedError, ServerError, and any other NetworkError
        print(f"Network error: {e}")

In [ ]:
# EAFP vs LBYL — a fundamental Python philosophy

# LBYL (Look Before You Leap) — the Swift way
# Check conditions before doing something

data = {"name": "Daniel", "age": 30}

# LBYL style (Swift-like):
if "email" in data:
    email = data["email"]
else:
    email = "not provided"
print(f"LBYL: {email}")

# EAFP style (Pythonic) — try it and handle the exception:
try:
    email = data["email"]
except KeyError:
    email = "not provided"
print(f"EAFP: {email}")

# Even more Pythonic — use .get() with a default:
email = data.get("email", "not provided")
print(f"Best: {email}")

# Why EAFP? In Python:
# - Exceptions are cheap (unlike some languages)
# - Avoids race conditions (file might disappear between check and use)
# - Often more readable for complex conditions
# - But don't use exceptions for normal control flow!

In [ ]:
# Exception chaining — preserve the original cause
# Like Swift's underlayingError pattern, but built into the language

class ConfigError(Exception):
    pass

def load_config(path: str) -> dict:
    try:
        with open(path) as f:
            import json
            return json.load(f)
    except FileNotFoundError as e:
        # 'from e' chains the exceptions — preserves the original cause
        raise ConfigError(f"Config file not found: {path}") from e
    except json.JSONDecodeError as e:
        raise ConfigError(f"Invalid JSON in config: {path}") from e

try:
    load_config("/nonexistent/config.json")
except ConfigError as e:
    print(f"Error: {e}")
    print(f"Caused by: {e.__cause__}")

---

## 6. Logging

Python's `logging` module is the standard way to emit log messages. It is more structured than `print()` and far more configurable than Swift's `print()` or even `os.log`.

### Swift comparison

```swift
// Swift: os.log (structured logging)
import os
let logger = Logger(subsystem: "com.app", category: "network")
logger.info("Request started")
logger.error("Request failed: \(error.localizedDescription)")
```

In [ ]:
import logging

# Basic setup — configure the root logger
# In a notebook, we need to configure each time since the root logger
# may already be configured.

# Create a logger (like Logger(subsystem:category:) in Swift)
logger = logging.getLogger("my_app")
logger.setLevel(logging.DEBUG)

# Remove existing handlers (notebook cleanup)
logger.handlers.clear()

# Add a console handler with formatting
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter(
    '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)
handler.setFormatter(formatter)
logger.addHandler(handler)

# Log levels (similar to os.log levels in Swift):
# DEBUG    — detailed diagnostic information
# INFO     — general operational messages
# WARNING  — something unexpected but not an error
# ERROR    — an error occurred but the app continues
# CRITICAL — a serious error, app may not continue

logger.debug("Detailed debug info — usually hidden in production")
logger.info("Application started successfully")
logger.warning("Disk space is getting low")
logger.error("Failed to connect to database")
logger.critical("Unrecoverable error — shutting down")

In [ ]:
import logging
from pathlib import Path

# Logging to a file (common in production)
log_path = Path(demo_dir) / "app.log"

file_logger = logging.getLogger("file_app")
file_logger.setLevel(logging.DEBUG)
file_logger.handlers.clear()

file_handler = logging.FileHandler(log_path)
file_handler.setLevel(logging.INFO)  # Only INFO and above go to file
file_handler.setFormatter(logging.Formatter(
    '%(asctime)s [%(levelname)s] %(name)s: %(message)s'
))

file_logger.addHandler(file_handler)

# Log some messages
file_logger.debug("This won't appear in the file (below INFO)")  
file_logger.info("Application initialized")
file_logger.warning("Using default configuration")
file_logger.error("Failed to load user preferences")

# Logging exceptions with traceback
try:
    result = 1 / 0
except ZeroDivisionError:
    file_logger.exception("Math error occurred")  # includes traceback!

# Show the log file
print(f"Log file contents ({log_path}):")
print(log_path.read_text())

In [ ]:
# Best practice: use __name__ as logger name
# This creates a hierarchy matching your module structure

import logging

# In a real module, __name__ would be something like 'myapp.utils.network'
# This lets you control logging at the package/module level

# Example of structured logging with extra context:
logger = logging.getLogger("my_app.network")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter('%(name)s - %(levelname)s - %(message)s'))
logger.addHandler(handler)

# f-string style (Python 3.12+) or %-style (traditional)
user_id = 42
endpoint = "/api/users"

# Traditional style (lazy evaluation — the string is only formatted if the message is logged)
logger.info("Request to %s for user %d", endpoint, user_id)

# f-string style (always formatted, even if log level would suppress it)
logger.info(f"Request to {endpoint} for user {user_id}")

# For performance-critical code, prefer %-style or check the level first:
if logger.isEnabledFor(logging.DEBUG):
    logger.debug(f"Expensive computation result: {sum(range(1000))}")

---

## Exercises

### Exercise 1: JSON Config Reader

Build a `ConfigReader` class that:
1. Loads a JSON config file
2. Supports dot-notation access (`config.get("database.host")`)
3. Supports default values
4. Validates required keys on load
5. Handles missing files and invalid JSON gracefully with custom exceptions

In [ ]:
# Exercise 1: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 1: Solution

import json
from pathlib import Path

class ConfigError(Exception):
    """Base exception for configuration errors."""
    pass

class ConfigFileError(ConfigError):
    """Error loading the config file."""
    pass

class ConfigValidationError(ConfigError):
    """Missing required configuration keys."""
    def __init__(self, missing_keys: list[str]):
        self.missing_keys = missing_keys
        super().__init__(f"Missing required keys: {', '.join(missing_keys)}")

_SENTINEL = object()  # unique sentinel for "no default"

class ConfigReader:
    """Read and access JSON config with dot-notation paths."""
    
    def __init__(self, path: str | Path, required_keys: list[str] | None = None):
        self._path = Path(path)
        self._data = self._load()
        if required_keys:
            self._validate(required_keys)
    
    def _load(self) -> dict:
        try:
            with open(self._path) as f:
                return json.load(f)
        except FileNotFoundError as e:
            raise ConfigFileError(f"Config file not found: {self._path}") from e
        except json.JSONDecodeError as e:
            raise ConfigFileError(f"Invalid JSON in {self._path}: {e}") from e
    
    def _validate(self, required_keys: list[str]) -> None:
        missing = [key for key in required_keys if self.get(key, _SENTINEL) is _SENTINEL]
        if missing:
            raise ConfigValidationError(missing)
    
    def get(self, dotted_key: str, default=_SENTINEL):
        """Get a value using dot-notation (e.g., 'database.host')."""
        keys = dotted_key.split(".")
        value = self._data
        for key in keys:
            if isinstance(value, dict) and key in value:
                value = value[key]
            elif default is not _SENTINEL:
                return default
            else:
                raise KeyError(f"Config key not found: {dotted_key}")
        return value
    
    def __repr__(self) -> str:
        return f"ConfigReader({self._path})"

# Create a sample config file
config_path = Path(demo_dir) / "app_config.json"
config_data = {
    "app_name": "MyApp",
    "version": "1.0.0",
    "database": {
        "host": "localhost",
        "port": 5432,
        "credentials": {
            "user": "admin",
            "password": "secret"
        }
    },
    "features": ["auth", "logging", "caching"]
}

with open(config_path, 'w') as f:
    json.dump(config_data, f, indent=2)

# Test it
config = ConfigReader(config_path, required_keys=["app_name", "database.host"])
print(f"App: {config.get('app_name')}")
print(f"DB Host: {config.get('database.host')}")
print(f"DB Port: {config.get('database.port')}")
print(f"DB User: {config.get('database.credentials.user')}")
print(f"Features: {config.get('features')}")
print(f"Missing key with default: {config.get('database.timeout', 30)}")

# Test error handling
try:
    config.get("nonexistent.key")
except KeyError as e:
    print(f"KeyError: {e}")

try:
    ConfigReader("/nonexistent/config.json")
except ConfigFileError as e:
    print(f"ConfigFileError: {e}")

### Exercise 2: CSV Data Processor

Build a CSV processor that:
1. Reads a CSV file with employee data
2. Filters by department
3. Calculates statistics (avg salary, min, max per department)
4. Writes a summary report as CSV
5. Handles errors (missing columns, invalid data)

In [ ]:
# Exercise 2: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 2: Solution

import csv
from pathlib import Path
from collections import defaultdict
import statistics

class CSVProcessingError(Exception):
    pass

class CSVProcessor:
    """Process CSV employee data with filtering and aggregation."""
    
    REQUIRED_COLUMNS = {"name", "department", "salary"}
    
    def __init__(self, filepath: str | Path):
        self.filepath = Path(filepath)
        self.rows: list[dict] = []
        self._load()
    
    def _load(self):
        """Load and validate CSV data."""
        try:
            with open(self.filepath, 'r') as f:
                reader = csv.DictReader(f)
                
                # Validate columns
                if reader.fieldnames is None:
                    raise CSVProcessingError("Empty CSV file")
                
                missing = self.REQUIRED_COLUMNS - set(reader.fieldnames)
                if missing:
                    raise CSVProcessingError(f"Missing columns: {missing}")
                
                # Load and validate rows
                for i, row in enumerate(reader, start=2):  # start=2 because row 1 is header
                    try:
                        row["salary"] = float(row["salary"])
                    except (ValueError, TypeError):
                        raise CSVProcessingError(
                            f"Invalid salary on row {i}: {row.get('salary')}"
                        )
                    self.rows.append(row)
                    
        except FileNotFoundError as e:
            raise CSVProcessingError(f"File not found: {self.filepath}") from e
    
    def filter_by_department(self, department: str) -> list[dict]:
        """Return rows matching the given department."""
        return [r for r in self.rows if r["department"] == department]
    
    def department_stats(self) -> dict[str, dict]:
        """Calculate salary statistics per department."""
        dept_salaries: dict[str, list[float]] = defaultdict(list)
        for row in self.rows:
            dept_salaries[row["department"]].append(row["salary"])
        
        stats = {}
        for dept, salaries in sorted(dept_salaries.items()):
            stats[dept] = {
                "count": len(salaries),
                "avg_salary": statistics.mean(salaries),
                "min_salary": min(salaries),
                "max_salary": max(salaries),
                "median_salary": statistics.median(salaries),
            }
        return stats
    
    def write_summary(self, output_path: str | Path):
        """Write department statistics to a CSV file."""
        stats = self.department_stats()
        output_path = Path(output_path)
        
        fieldnames = ["department", "count", "avg_salary", "min_salary", "max_salary", "median_salary"]
        with open(output_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for dept, dept_stats in stats.items():
                row = {"department": dept, **dept_stats}
                row["avg_salary"] = f"{row['avg_salary']:.2f}"
                writer.writerow(row)
        
        return output_path

# Create test data
test_csv = Path(demo_dir) / "employees_full.csv"
with open(test_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["name", "department", "salary"])
    writer.writerows([
        ["Alice", "Engineering", 130000],
        ["Bob", "Engineering", 125000],
        ["Charlie", "Engineering", 140000],
        ["Diana", "Design", 95000],
        ["Eve", "Design", 105000],
        ["Frank", "Marketing", 88000],
        ["Grace", "Marketing", 92000],
        ["Hank", "Marketing", 85000],
    ])

# Process
processor = CSVProcessor(test_csv)

# Filter
engineers = processor.filter_by_department("Engineering")
print("Engineers:")
for emp in engineers:
    print(f"  {emp['name']}: ${emp['salary']:,.0f}")

# Stats
print("\nDepartment Statistics:")
for dept, stats in processor.department_stats().items():
    print(f"  {dept}: {stats['count']} people, avg ${stats['avg_salary']:,.0f}")

# Write summary
summary_path = processor.write_summary(Path(demo_dir) / "summary.csv")
print(f"\nSummary written to {summary_path}:")
print(summary_path.read_text())

### Exercise 3: Custom Exception Hierarchy

Build an exception hierarchy for a REST API client:
1. Base `APIError` with status code and message
2. `ClientError` (4xx) with subtypes: `NotFoundError` (404), `ValidationError` (422), `RateLimitError` (429)
3. `ServerError` (5xx) with `InternalServerError` (500) and `ServiceUnavailableError` (503)
4. A helper function `raise_for_status(status_code, body)` that raises the appropriate exception
5. Each exception should have useful attributes (status_code, response_body, retry_after for rate limits)

In [ ]:
# Exercise 3: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 3: Solution

class APIError(Exception):
    """Base exception for API errors."""
    def __init__(self, status_code: int, message: str, response_body: str = ""):
        self.status_code = status_code
        self.response_body = response_body
        super().__init__(f"HTTP {status_code}: {message}")

class ClientError(APIError):
    """4xx client errors."""
    pass

class NotFoundError(ClientError):
    """404 Not Found."""
    def __init__(self, resource: str, response_body: str = ""):
        self.resource = resource
        super().__init__(404, f"Resource not found: {resource}", response_body)

class ValidationError(ClientError):
    """422 Unprocessable Entity."""
    def __init__(self, errors: dict, response_body: str = ""):
        self.errors = errors
        error_summary = "; ".join(f"{k}: {v}" for k, v in errors.items())
        super().__init__(422, f"Validation failed: {error_summary}", response_body)

class RateLimitError(ClientError):
    """429 Too Many Requests."""
    def __init__(self, retry_after: int = 60, response_body: str = ""):
        self.retry_after = retry_after
        super().__init__(429, f"Rate limited. Retry after {retry_after}s", response_body)

class ServerError(APIError):
    """5xx server errors."""
    pass

class InternalServerError(ServerError):
    """500 Internal Server Error."""
    def __init__(self, response_body: str = ""):
        super().__init__(500, "Internal server error", response_body)

class ServiceUnavailableError(ServerError):
    """503 Service Unavailable."""
    def __init__(self, retry_after: int | None = None, response_body: str = ""):
        self.retry_after = retry_after
        msg = "Service unavailable"
        if retry_after:
            msg += f" (retry after {retry_after}s)"
        super().__init__(503, msg, response_body)

def raise_for_status(status_code: int, body: str = "", headers: dict | None = None):
    """Raise the appropriate exception for an HTTP status code."""
    headers = headers or {}
    
    if 200 <= status_code < 400:
        return  # success, no exception
    
    if status_code == 404:
        raise NotFoundError(body, body)
    elif status_code == 422:
        import json
        try:
            errors = json.loads(body)
        except (json.JSONDecodeError, TypeError):
            errors = {"detail": body}
        raise ValidationError(errors, body)
    elif status_code == 429:
        retry_after = int(headers.get("Retry-After", 60))
        raise RateLimitError(retry_after, body)
    elif status_code == 500:
        raise InternalServerError(body)
    elif status_code == 503:
        retry_after = headers.get("Retry-After")
        raise ServiceUnavailableError(
            int(retry_after) if retry_after else None, body
        )
    elif 400 <= status_code < 500:
        raise ClientError(status_code, body, body)
    elif 500 <= status_code < 600:
        raise ServerError(status_code, body, body)

# Test the hierarchy
test_cases = [
    (200, "OK", {}),
    (404, "/api/users/999", {}),
    (422, '{"email": "invalid format", "age": "must be positive"}', {}),
    (429, "Too many requests", {"Retry-After": "30"}),
    (500, "Internal error", {}),
    (503, "Maintenance", {"Retry-After": "300"}),
]

for status, body, headers in test_cases:
    try:
        raise_for_status(status, body, headers)
        print(f"  {status}: OK (no exception)")
    except RateLimitError as e:
        print(f"  {status}: {e} (retry after {e.retry_after}s)")
    except ClientError as e:
        print(f"  {status}: [Client] {e}")
    except ServerError as e:
        print(f"  {status}: [Server] {e}")

### Exercise 4: Log File Analyzer

Build a log analyzer that:
1. Reads a log file line by line (memory efficient)
2. Parses each line to extract timestamp, level, and message
3. Counts occurrences of each log level
4. Finds the most common error messages
5. Writes a report to a JSON file

In [ ]:
# Exercise 4: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 4: Solution

import json
import re
from collections import Counter
from pathlib import Path
from datetime import datetime

class LogAnalyzer:
    """Analyze log files and produce summary reports."""
    
    # Pattern: 2025-01-15 14:30:00 [ERROR] Some message here
    LOG_PATTERN = re.compile(
        r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) '
        r'\[(\w+)\] '
        r'(.+)'
    )
    
    def __init__(self):
        self.level_counts: Counter = Counter()
        self.error_messages: Counter = Counter()
        self.total_lines = 0
        self.parse_errors = 0
        self.first_timestamp: str | None = None
        self.last_timestamp: str | None = None
    
    def analyze(self, filepath: str | Path) -> "LogAnalyzer":
        """Analyze a log file line by line (memory efficient)."""
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                self.total_lines += 1
                match = self.LOG_PATTERN.match(line)
                
                if match:
                    timestamp, level, message = match.groups()
                    
                    if self.first_timestamp is None:
                        self.first_timestamp = timestamp
                    self.last_timestamp = timestamp
                    
                    self.level_counts[level] += 1
                    
                    if level in ("ERROR", "CRITICAL"):
                        self.error_messages[message] += 1
                else:
                    self.parse_errors += 1
        
        return self
    
    def top_errors(self, n: int = 5) -> list[tuple[str, int]]:
        """Return the n most common error messages."""
        return self.error_messages.most_common(n)
    
    def write_report(self, output_path: str | Path):
        """Write analysis report as JSON."""
        report = {
            "summary": {
                "total_lines": self.total_lines,
                "parse_errors": self.parse_errors,
                "time_range": {
                    "first": self.first_timestamp,
                    "last": self.last_timestamp,
                },
            },
            "level_counts": dict(self.level_counts.most_common()),
            "top_errors": [
                {"message": msg, "count": count}
                for msg, count in self.top_errors(10)
            ],
        }
        
        output_path = Path(output_path)
        with open(output_path, 'w') as f:
            json.dump(report, f, indent=2)
        
        return output_path

# Create a sample log file
import random
random.seed(42)

log_file = Path(demo_dir) / "application.log"
levels = ["DEBUG"] * 5 + ["INFO"] * 10 + ["WARNING"] * 3 + ["ERROR"] * 2 + ["CRITICAL"] * 1
error_msgs = [
    "Connection refused to database",
    "Timeout waiting for response",
    "Invalid authentication token",
    "Out of memory",
]
info_msgs = [
    "Request processed successfully",
    "Cache hit for key",
    "User logged in",
    "Health check passed",
]

with open(log_file, 'w') as f:
    base = datetime(2025, 1, 15, 8, 0, 0)
    for i in range(100):
        ts = base + __import__('datetime').timedelta(seconds=i * 30)
        level = random.choice(levels)
        if level in ("ERROR", "CRITICAL"):
            msg = random.choice(error_msgs)
        else:
            msg = random.choice(info_msgs)
        f.write(f"{ts.strftime('%Y-%m-%d %H:%M:%S')} [{level}] {msg}\n")

# Analyze
analyzer = LogAnalyzer().analyze(log_file)

print(f"Total lines: {analyzer.total_lines}")
print(f"Parse errors: {analyzer.parse_errors}")
print(f"\nLevel counts:")
for level, count in analyzer.level_counts.most_common():
    print(f"  {level}: {count}")
print(f"\nTop errors:")
for msg, count in analyzer.top_errors(5):
    print(f"  [{count}x] {msg}")

# Write report
report_path = analyzer.write_report(Path(demo_dir) / "log_report.json")
print(f"\nReport written to {report_path}")

### Exercise 5: File Watcher Utility

Build a utility that monitors a directory for changes:
1. Takes a directory path and file pattern (e.g., `*.json`)
2. Tracks file sizes, modification times, and checksums
3. Detects added, modified, and deleted files
4. Uses pathlib throughout
5. Logs changes using the logging module

In [ ]:
# Exercise 5: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 5: Solution

import hashlib
import logging
from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class FileInfo:
    """Metadata about a tracked file."""
    path: Path
    size: int
    modified: float
    checksum: str

@dataclass
class ChangeReport:
    """Report of detected changes."""
    added: list[Path] = field(default_factory=list)
    modified: list[Path] = field(default_factory=list)
    deleted: list[Path] = field(default_factory=list)
    
    @property
    def has_changes(self) -> bool:
        return bool(self.added or self.modified or self.deleted)

class FileWatcher:
    """Monitor a directory for file changes."""
    
    def __init__(self, directory: str | Path, pattern: str = "*"):
        self.directory = Path(directory)
        self.pattern = pattern
        self._tracked: dict[Path, FileInfo] = {}
        
        # Set up logging
        self.logger = logging.getLogger(f"FileWatcher({self.directory.name})")
        self.logger.setLevel(logging.DEBUG)
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(logging.Formatter('%(name)s - %(levelname)s - %(message)s'))
            self.logger.addHandler(handler)
        
        if not self.directory.is_dir():
            raise FileNotFoundError(f"Directory not found: {self.directory}")
    
    @staticmethod
    def _checksum(filepath: Path) -> str:
        """Calculate MD5 checksum of a file."""
        hasher = hashlib.md5()
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                hasher.update(chunk)
        return hasher.hexdigest()
    
    def _get_file_info(self, filepath: Path) -> FileInfo:
        """Get metadata for a file."""
        stat = filepath.stat()
        return FileInfo(
            path=filepath,
            size=stat.st_size,
            modified=stat.st_mtime,
            checksum=self._checksum(filepath),
        )
    
    def snapshot(self) -> dict[Path, FileInfo]:
        """Take a snapshot of current files matching the pattern."""
        current = {}
        for filepath in self.directory.glob(self.pattern):
            if filepath.is_file():
                current[filepath] = self._get_file_info(filepath)
        return current
    
    def scan(self) -> ChangeReport:
        """Scan for changes since last scan."""
        current = self.snapshot()
        report = ChangeReport()
        
        # Check for added and modified files
        for path, info in current.items():
            if path not in self._tracked:
                report.added.append(path)
                self.logger.info("Added: %s (%d bytes)", path.name, info.size)
            elif self._tracked[path].checksum != info.checksum:
                report.modified.append(path)
                old_size = self._tracked[path].size
                self.logger.warning(
                    "Modified: %s (size: %d -> %d bytes)",
                    path.name, old_size, info.size
                )
        
        # Check for deleted files
        for path in self._tracked:
            if path not in current:
                report.deleted.append(path)
                self.logger.error("Deleted: %s", path.name)
        
        # Update tracked state
        self._tracked = current
        
        if not report.has_changes:
            self.logger.debug("No changes detected")
        
        return report

# Demo
watch_dir = Path(demo_dir) / "watched"
watch_dir.mkdir(exist_ok=True)

# Create initial files
(watch_dir / "config.json").write_text('{"version": 1}')
(watch_dir / "data.json").write_text('{"items": []}')

# Create watcher and do initial scan
watcher = FileWatcher(watch_dir, "*.json")
print("=== Initial scan ===")
report = watcher.scan()
print(f"Added: {len(report.added)}, Modified: {len(report.modified)}, Deleted: {len(report.deleted)}")

# Modify a file
(watch_dir / "config.json").write_text('{"version": 2}')
# Add a file
(watch_dir / "new_file.json").write_text('{"new": true}')
# Delete a file
(watch_dir / "data.json").unlink()

print("\n=== Second scan (after changes) ===")
report = watcher.scan()
print(f"Added: {len(report.added)}, Modified: {len(report.modified)}, Deleted: {len(report.deleted)}")

print("\n=== Third scan (no changes) ===")
report = watcher.scan()
print(f"Added: {len(report.added)}, Modified: {len(report.modified)}, Deleted: {len(report.deleted)}")

In [ ]:
# Cleanup: remove the temporary demo directory
import shutil
shutil.rmtree(demo_dir)
print(f"Cleaned up {demo_dir}")

---

## Key Takeaways

1. **File I/O**: Always use `with open(...)` for automatic cleanup. Use `pathlib.Path` instead of `os.path` for path operations.

2. **pathlib**: Python's `Path` is more ergonomic than Swift's `URL`/`FileManager` combination. The `/` operator for joining paths is particularly clean.

3. **JSON**: Python's `json` module works with dicts (dynamic), unlike Swift's Codable (static). Use `dataclasses` + `asdict()` for a Codable-like experience.

4. **CSV**: Python has built-in CSV support. `DictReader`/`DictWriter` are the most convenient for typical use cases.

5. **Exceptions**: Python uses class hierarchies (not enums) for errors. Embrace EAFP (try/except) over LBYL (check first). Use `from e` for exception chaining.

6. **Logging**: Use the `logging` module, not `print()`, for any non-trivial application. Configure it once at application startup.

---

*Next up: Notebook 6 -- The Python Ecosystem*